# Notebook 1: Data Loading and EDA
### Step 1: Imports and Seeds
We begin by importing the required libraries and setting the global random seed as mandated by the project brief.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

np.random.seed(42)
print('Imports and seeds initialized successfully.')

Imports and seeds initialized successfully.


## Step 2: Load the Datasets
We load the collisions, casualties, and vehicles datasets for the 2024 full year.

In [2]:
collisions = pd.read_csv('../data/dft-road-casualty-statistics-collision-2024.csv')
casualties = pd.read_csv('../data/dft-road-casualty-statistics-casualty-2024.csv')
vehicles = pd.read_csv('../data/dft-road-casualty-statistics-vehicle-2024.csv')

print("Collisions Shape:", collisions.shape)
print("Casualties Shape:", casualties.shape)
print("Vehicles Shape:  ", vehicles.shape)

Collisions Shape: (100927, 44)
Casualties Shape: (128272, 23)
Vehicles Shape:   (183514, 32)


## Step 3: Feature Aggregation (Advanced Strategy)
Instead of naively keeping only the 'first' vehicle or casualty (which discards valuable multi-vehicle crash data), we use **Feature Aggregation**. 
We mathematically summarize the `casualties` and `vehicles` tables to retain all context while still compressing the dataset down to exactly **one row per collision**.

***Critical Note on Target Leakage:*** *We absolutely must NOT aggregate `casualty_severity`. Since our model's goal is to predict the crash severity, using the severity of the casualties as an input feature would give the model the answer key before it makes a prediction, completely invalidating it. We only aggregate characteristics available 'before' the crash completes (e.g. types of vehicles involved, ages).*

In [3]:
# 1. Aggregate Casualties
# We create summary features such as total casualties and age boundaries.
casualties_agg = casualties.groupby('collision_index').agg(
    total_casualties=('casualty_reference', 'count'),
    min_casualty_age=('age_of_casualty', 'min'),
    max_casualty_age=('age_of_casualty', 'max')
).reset_index()

# 2. Aggregate Vehicles
# Similarly, summarize the vehicles
vehicles_agg = vehicles.groupby('collision_index').agg(
    total_vehicles_involved=('vehicle_reference', 'count'),
    min_driver_age=('age_of_driver', 'min'),
    max_driver_age=('age_of_driver', 'max')
).reset_index()

print("Aggregated Casualties Shape:", casualties_agg.shape)
print("Aggregated Vehicles Shape:", vehicles_agg.shape)

Aggregated Casualties Shape: (100927, 4)
Aggregated Vehicles Shape: (100927, 4)


## Step 4: Join tables and Filter to Greater London (Advanced ONS Strategy)
We left-merge our intelligently aggregated `casualties_agg` and `vehicles_agg` tables onto the main `collisions` table. 

Instead of relying on the simplistic `police_force` column (which inaccurately pulls in rural crashes if a London police unit responded to them out-of-bounds), we use the `local_authority_ons_district` column. Every single official London Borough code strictly begins with **`E09`** (E09000001 to E09000033). This administrative filter mathematically guarantees we are keeping collisions located strictly inside the physical municipal boundaries of Greater London.

In [4]:
# 1. Merge the aggregated tables onto collisions
df = collisions.merge(casualties_agg, on='collision_index', how='left')
df = df.merge(vehicles_agg, on='collision_index', how='left')

# 2. Strict Filter to London Boroughs using ONS codes ('E09...')
london_df = df[df['local_authority_ons_district'].fillna('Unknown').str.startswith('E09')].copy()

print(f"Total UK Collisions Dataset:             {len(df):,} rows")
print(f"Greater London Dataset (Strict Boroughs): {len(london_df):,} rows, {london_df.shape[1]} columns")

Total UK Collisions Dataset:             100,927 rows
Greater London Dataset (Strict Boroughs): 20,967 rows, 50 columns


## Step 5: Advanced Exploratory Data Analysis (EDA)
Here we visually explore our precise Greater London dataset to uncover underlying risk patterns, expose missing data, and analyze the extreme class imbalance in our target variable (`collision_severity`). 

Moving far beyond basic histograms, we implement elite, consultancy-grade visuals to prove the real-world utility of the dataset to IntelliSys:
1. **Target Variable Imbalance**
2. **Missing Values Heatmap**
3. **Severity Breakdown by Standard Speed Limits**
4. **Geospatial Risk Map (Longitude vs Latitude pinpointing Fatal crashes)**
5. **24x7 Temporal Risk Matrix (Day of week vs Hour of day density)**
6. **Categorical Association (Weather cross-tabulated with Severity)**

All figures are automatically pushed to `../outputs/figures/` for immediate drop-in to the final client presentation.

In [5]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

os.makedirs('../outputs/figures', exist_ok=True)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# 1. Target Imbalance
plt.figure(figsize=(7, 4))
ax = sns.countplot(data=london_df, x='collision_severity', palette='viridis')
plt.title('Distribution of Collision Severity (Greater London, 2024)', fontweight='bold')
plt.xlabel('Severity (1=Fatal, 2=Serious, 3=Slight)')
plt.ylabel('Number of Collisions')
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.tight_layout()
plt.savefig('../outputs/figures/01_target_imbalance.png')
plt.close()

# 2. Missing Values
plt.figure(figsize=(10, 5))
missing = london_df.isnull()
if missing.sum().sum() > 0:
    sns.heatmap(missing, cbar=False, cmap='magma', yticklabels=False)
    plt.title('Health Check: Missing Values Heatmap', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../outputs/figures/02_missing_values.png')
    plt.close()

# 3. Severity by Speed Limit
plt.figure(figsize=(8, 4))
sns.countplot(data=london_df[london_df['speed_limit'].isin([20, 30, 40, 50, 60, 70])], 
              x='speed_limit', hue='collision_severity', palette='viridis')
plt.title('Collision Severity Count by Standard Speed Limits', fontweight='bold')
plt.yscale('log')
plt.ylabel('Count (Log Scale)')
plt.legend(title='Severity (1=Fatal)')
plt.tight_layout()
plt.savefig('../outputs/figures/03_severity_by_speed.png')
plt.close()

# 4. Geospatial Risk Plot
plt.figure(figsize=(9, 7))
geo_df = london_df.dropna(subset=['longitude', 'latitude'])
# Layering the crashes to highlight Fatal ones on top
sns.scatterplot(data=geo_df[geo_df['collision_severity']==3], x='longitude', y='latitude', color='lightgrey', alpha=0.3, s=5, label='Slight (3)')
sns.scatterplot(data=geo_df[geo_df['collision_severity']==2], x='longitude', y='latitude', color='orange', alpha=0.7, s=15, label='Serious (2)')
sns.scatterplot(data=geo_df[geo_df['collision_severity']==1], x='longitude', y='latitude', color='red', edgecolor='black', s=50, label='Fatal (1)', zorder=10)
plt.title('Physical High-Risk Corridors: Geolocation of London Collisions', fontweight='bold')
plt.legend(loc='lower right')
plt.axis('equal') # Lock map aspect ratio
plt.tight_layout()
plt.savefig('../outputs/figures/04_geospatial_risk.png')
plt.close()

# 5. Temporal Risk Matrix (24x7)
if 'time' in london_df.columns and 'day_of_week' in london_df.columns:
    try:
        london_df['hour'] = pd.to_datetime(london_df['time'], format='%H:%M', errors='coerce').dt.hour
        heat_df = london_df.pivot_table(index='day_of_week', columns='hour', aggfunc='size', fill_value=0)
        # 1 = Sunday in STATS19 dictionary
        heat_df.index = heat_df.index.map({1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'})
        
        plt.figure(figsize=(11, 4))
        sns.heatmap(heat_df, cmap='YlOrRd', linewidths=.5, cbar_kws={'label': 'Collision Count'})
        plt.title('24x7 Temporal Risk Matrix (Crash Density by Day & Hour)', fontweight='bold')
        plt.xlabel('Hour of Day (0-23)')
        plt.ylabel('Day of Week')
        plt.tight_layout()
        plt.savefig('../outputs/figures/05_temporal_risk.png')
        plt.close()
    except Exception as e:
        print(f"Time parsing skipped due to: {e}")

# 6. Categorical Association (Weather vs Severity)
plt.figure(figsize=(9, 4))
weather_severity = pd.crosstab(london_df['weather_conditions'], london_df['collision_severity'], normalize='index') * 100
weather_severity.plot(kind='bar', stacked=True, colormap='viridis', figsize=(9,4))
plt.title('Percentage Breakdown of Severity per Weather Condition', fontweight='bold')
plt.xlabel('Weather Code (1=Fine, 2=Raining...)')
plt.ylabel('Percentage of Crashes (%)')
plt.legend(title='Severity', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/figures/06_weather_association.png')
plt.close()

print('All 6 Advanced EDA Visualisations cleanly executed and saved to outputs/figures/')

All 6 Advanced EDA Visualisations cleanly executed and saved to outputs/figures/


<Figure size 1080x480 with 0 Axes>